# model_test_prototype

Version prototype de `model_test`, alignée sur `graph_pipeline_prototype`.

Ce qui change par rapport à `model_test` :

- **Un seul chemin d'extraction des triggers.** Ils viennent de `prod_results_val_prototype`, qui couvre *toutes* les exécutions du split. `extract_triggers` (qui partait de `val_split["edge"]`, donc des seules exécutions avec positifs) et `extract_negative_only_triggers` sont fusionnés en `extract_all_triggers`.
- **Une seule passe d'inférence**, au lieu de deux (population avec positifs puis population full-negatives).
- **Artefacts isolés** : `data_prototype.pt`, `node2idx_prototype.json`, `exec_mappings_prototype.json`. Les `exec_code` du prototype diffèrent du pipeline principal, il ne faut donc surtout pas croiser les deux jeux de fichiers.
- **Le display « production » est lu depuis le parquet**, dans ses deux variantes `_display` et `_relevance`. L'ancienne fonction `get_production_candidates`, qui re-requêtait `ads_adlog_fct` avec un `Window.orderBy` global sans `partitionBy` — et donnait donc un classement qui ne correspondait pas à celui utilisé pour construire les edges — est supprimée.
- **`positive_ranks_model` disparaît** : il dépendait des positifs portés par le tenseur d'edges. Le MRR du modèle se calcule maintenant à partir des `rank` de `positives`, exactement comme pour les deux variantes prod — c'est ce qui rend la comparaison symétrique dans `model_stat_prototype`.

In [ ]:
%pip install /Workspace/Users/neil.braun@mirakl.com/.bundle/fast-gnn-benchmark/dev/files
dbutils.library.restartPython()

In [ ]:
from fast_gnn_benchmark.trainer import load_model_from_checkpoint
from fast_gnn_benchmark.data.dataset.coview_mdm import CoViewMDMDataset

import os
import torch
import boto3, json, io
import pyspark.sql.functions as F
from pyspark.sql import DataFrame, Row
from IPython.display import display, HTML

In [ ]:
from datetime import datetime

# Dossier propre au prototype: gae_gcn_coview_mdm_prototype.yml y ecrit ses checkpoints, pour ne pas
# polluer coview-mdm-128 que model_test.ipynb (non-prototype) lit toujours.
ckpt_dir = "/dbfs/tmp/nbraun/checkpoints/coview-mdm-prototype"

files = [
    (os.path.getmtime(os.path.join(ckpt_dir, f)), f)
    for f in os.listdir(ckpt_dir)
    if f.endswith(".ckpt")
]

for mtime, fname in sorted(files, reverse=True):
    print(f"{datetime.fromtimestamp(mtime):%Y-%m-%d %H:%M:%S}  {fname}")

In [ ]:
from fast_gnn_benchmark.models.link_prediction import LinkPredictionModel

device = "cuda" if torch.cuda.is_available() else "cpu"

# save_top_k=1 sur val/mrr_trigger: le dossier ne contient que le checkpoint du meilleur epoch. On le
# resout dynamiquement, son nom dependant de l'epoch atteint -- pas de nom de fichier en dur ici.
assert files, f"aucun .ckpt dans {ckpt_dir}: lancer l'entrainement avec gae_gcn_coview_mdm_prototype.yml"
checkpoint_path = os.path.join(ckpt_dir, sorted(files, reverse=True)[0][1])
print(f"checkpoint: {checkpoint_path}")

raw_ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

model = LinkPredictionModel.load_from_checkpoint(checkpoint_path, map_location=device, weights_only=False)
model.eval()
model.to(device)

print("epoch:", raw_ckpt["epoch"])
print("global_step:", raw_ckpt["global_step"])
assert f"epoch={raw_ckpt['epoch']}-step={raw_ckpt['global_step']}" in checkpoint_path

print(model.hparams.model_parameters)

for cb_state in raw_ckpt["callbacks"].values():
    if "best_model_score" in cb_state:
        print("best_model_score (val/mrr_trigger):", cb_state["best_model_score"])

print("nb params:", sum(p.numel() for p in model.parameters()))

## Chargement du dataset et des mappings prototype

In [ ]:
BUCKET = "mirakl-data-science-tmp2"
PREFIX = "nbraun/datasets/coview-mdm"

# Artefacts prototype: exec2code y couvre toutes les executions du split, donc les exec_code
# diffèrent de ceux du pipeline principal. Ne jamais melanger avec data.pt / exec_mappings.json.
dataset = CoViewMDMDataset(bucket=BUCKET, s3_key=f"{PREFIX}/data_prototype.pt")

s3 = boto3.client("s3")

node2idx_raw = json.loads(
    s3.get_object(Bucket=BUCKET, Key=f"{PREFIX}/node2idx_prototype.json")["Body"].read()
)
node2idx = {int(k): v for k, v in node2idx_raw.items()}
idx2node = {v: k for k, v in node2idx.items()}

exec_mappings = json.loads(
    s3.get_object(Bucket=BUCKET, Key=f"{PREFIX}/exec_mappings_prototype.json")["Body"].read()
)

print(f"num_nodes: {dataset.num_nodes}")
print(f"node2idx entries: {len(node2idx)}")
for split in ["train", "val", "test"]:
    print(f"{split}: {len(exec_mappings[split]['code2exec'])} executions")

## Tables prototype

`prod_results_val_prototype` porte les deux variantes du retour prod pour chaque trigger : `_display` (les 12 premiers dans l'ordre d'affichage) et `_relevance` (les 12 premiers par `relevanceScore`). C'est aussi elle qui définit la population de triggers à scorer.

In [ ]:
prod_results_val_prototype = spark.read.parquet(
    f"s3://{BUCKET}/{PREFIX}/prod_results_val_prototype.parquet"
)
sessions_raw_val_prototype = spark.read.parquet(
    f"s3://{BUCKET}/{PREFIX}/sessions_raw_val_prototype.parquet"
)

print(f"prod_results_val_prototype: {prod_results_val_prototype.count()} triggers")
prod_results_val_prototype.printSchema()
display(prod_results_val_prototype.limit(1))

print(f"sessions_raw_val_prototype: {sessions_raw_val_prototype.count()} triggers")
display(sessions_raw_val_prototype.limit(1))

## Embeddings de nœuds, calculés une seule fois et réutilisés par tous les triggers

In [ ]:
import torch.nn.functional as TF
from torch_geometric.nn.conv.gcn_conv import gcn_norm

N = dataset.num_nodes
edge_index = dataset.data.edge_index.to(device)

norm_edge_index, norm_edge_weight = gcn_norm(edge_index, num_nodes=N, add_self_loops=True)
adj_norm = torch.sparse_coo_tensor(norm_edge_index, norm_edge_weight, (N, N)).coalesce()


def sparse_backbone_forward(backbone, x, adj_norm):
    conv_layers = backbone.conv_layers
    for layer_index, conv in enumerate(conv_layers):
        x = conv.lin(x)
        x = torch.sparse.mm(adj_norm, x)
        if conv.bias is not None:
            x = x + conv.bias
        if layer_index != len(conv_layers) - 1:
            x = TF.relu(x)
    return x


batch_size = 8192

with torch.no_grad():
    x = model.model.embedder(dataset.data.x.to(device))
    x = sparse_backbone_forward(model.model.backbone, x, adj_norm)

print(f"x shape: {tuple(x.shape)}")

## Extraction des triggers — un seul chemin

Tous les triggers de `prod_results_val_prototype`, sans distinction entre exécutions avec positifs et exécutions full-negatives.

In [ ]:
def extract_all_triggers(prod_results: DataFrame) -> list[dict]:
    """Tous les triggers de la table prod prototype.

    Remplace extract_triggers (qui partait de val_split["edge"], donc des seules executions ayant
    au moins un positif dans le top-12 prod) et extract_negative_only_triggers. Le tenseur d'edges
    n'est plus utilise ici: la population d'analyse est definie independamment de l'etiquetage.
    """
    rows = (
        prod_results
        .select("exec_code", "trigger_internal_id")
        .dropDuplicates(["exec_code"])
        .collect()
    )

    triggers = []
    skipped_no_node = 0

    for row in rows:
        trigger_internal_id = int(row["trigger_internal_id"])

        if trigger_internal_id not in node2idx:
            skipped_no_node += 1
            continue

        triggers.append({
            "exec_code": row["exec_code"],
            "trigger_internal_id": trigger_internal_id,
            "trigger_node_id": node2idx[trigger_internal_id],
        })

    print(f"triggers hors graphe (produit sans node_id): {skipped_no_node}/{len(rows)}")

    return triggers


triggers = extract_all_triggers(prod_results_val_prototype)
print(f"{len(triggers)} triggers a scorer")
print(triggers[0])

In [ ]:
def extract_categories(triggers: list[dict]) -> list[dict]:
    """Attache a chaque trigger sa t2s_best_fitting_category. trigger_internal_id est deja
    renseigne par extract_all_triggers, contrairement a la version non-prototype."""

    unique_internal_ids = {t["trigger_internal_id"] for t in triggers}

    df_trigger_ids = spark.createDataFrame(
        [(str(i),) for i in unique_internal_ids], schema="internalId string"
    )

    df_categories = (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(F.col("customer_short_name") == "maisons-du-monde")
        .join(F.broadcast(df_trigger_ids), on="internalId", how="left_semi")
        .select("internalId", F.col("t2s_best_fitting_category")[0].alias("category"))
        .dropDuplicates(["internalId"])
        .collect()
    )

    category_by_internal_id = {
        int(row["internalId"]): row["category"]
        for row in df_categories
        if row["category"] is not None
    }

    for trigger in triggers:
        trigger["category"] = category_by_internal_id.get(trigger["trigger_internal_id"])

    return triggers


triggers = extract_categories(triggers)
missing = sum(1 for t in triggers if t["category"] is None)
print(f"triggers sans categorie: {missing}/{len(triggers)}")
print(triggers[0])

In [ ]:
def build_candidates_by_category(triggers: list[dict]) -> dict:

    unique_categories = {t["category"] for t in triggers if t["category"] is not None}

    df_category_products = (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(
            F.col("t2s_best_fitting_category")[0].isin(list(unique_categories))
            & (F.col("customer_short_name") == "maisons-du-monde")
        )
        .select("internalId", F.col("t2s_best_fitting_category")[0].alias("category"))
        .dropDuplicates(["internalId"])
        .collect()
    )

    internal_ids_by_category = {c: [] for c in unique_categories}
    for row in df_category_products:
        internal_ids_by_category[row["category"]].append(int(row["internalId"]))

    candidate_node_ids_by_category = {}
    for category, internal_ids in internal_ids_by_category.items():
        node_ids = torch.tensor([node2idx[i] for i in internal_ids if i in node2idx])
        candidate_node_ids_by_category[category] = torch.unique(node_ids)  # plusieurs internalId peuvent partager le même node_id

    return candidate_node_ids_by_category

candidates_by_category = build_candidates_by_category(triggers)
pool_sizes = [c.numel() for c in candidates_by_category.values()]
print(f"nombre de catégories: {len(candidates_by_category)}")
print(f"taille moyenne du pool de candidats par catégorie: {sum(pool_sizes) / len(pool_sizes):.0f}")


## Inférence sur tous les triggers

Le modèle classe l'intégralité du pool de la catégorie du trigger, puis on garde son top 12. Ce pool est bien plus large que les 12 candidats de la prod : c'est ce qui permet au modèle de remonter un produit que la prod n'avait pas proposé du tout.

In [ ]:
TOP_K = 12


def run_inference(triggers: list[dict], candidates_by_category: dict) -> list[dict]:

    skipped_no_candidates = 0

    with torch.no_grad():
        for trigger in triggers:
            if trigger["category"] is None:
                skipped_no_candidates += 1
                continue

            candidates_to_score = candidates_by_category[trigger["category"]]
            candidates_to_score = candidates_to_score[candidates_to_score != trigger["trigger_node_id"]]

            if candidates_to_score.numel() == 0:
                skipped_no_candidates += 1
                continue

            target_edges = torch.stack([
                torch.full_like(candidates_to_score, trigger["trigger_node_id"]),
                candidates_to_score,
            ])

            logits_chunks = []
            for start in range(0, target_edges.shape[1], batch_size):
                chunk = target_edges[:, start:start + batch_size].to(device)
                logits_chunks.append(model.model.classifier(x, x, chunk))

            # topk sur le device: seuls TOP_K logits et indices traversent le lien GPU -> CPU, la ou
            # un argsort complet suivi de trois .tolist() rapatriait et allouait candidate_pool_size
            # elements Python par trigger. Possible uniquement depuis que positive_ranks_model a
            # disparu: plus personne n'a besoin du classement complet, seuls les 12 premiers.
            logits = torch.cat(logits_chunks)
            k = min(TOP_K, logits.numel())  # un pool peut compter moins de 12 produits
            top_logits, top_positions = torch.topk(logits, k)
            top_node_ids = candidates_to_score[top_positions.cpu()]

            top12_model = [
                {
                    "internalId": int(idx2node[node_id]),
                    "score": float(score),
                    "rank": rank,
                }
                for rank, (node_id, score) in enumerate(
                    zip(top_node_ids.tolist(), top_logits.cpu().tolist()), start=1
                )
            ]

            trigger["candidate_pool_size"] = candidates_to_score.numel()
            trigger["top12_model"] = top12_model

    print(f"triggers non scores (categorie manquante ou pool vide): {skipped_no_candidates}/{len(triggers)}")

    return triggers

In [ ]:
import copy
from torch.profiler import profile, ProfilerActivity

SAMPLE_SIZE = 20
triggers_sample = copy.deepcopy(triggers[:SAMPLE_SIZE])

run_inference(triggers_sample, candidates_by_category)
torch.cuda.synchronize()

triggers_sample = copy.deepcopy(triggers[:SAMPLE_SIZE])

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    run_inference(triggers_sample, candidates_by_category)

torch.cuda.synchronize()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=20))
print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=20))

In [ ]:
triggers = run_inference(triggers, candidates_by_category)

n_scored = sum(1 for t in triggers if t.get("top12_model"))
print(f"{n_scored}/{len(triggers)} triggers scores")
print(triggers[0].get("top12_model", [])[:3])

## Résultats du modèle

Même structure que les variantes prod : `products_returned`, `positives`, `negatives`. Les positifs sont les produits du top-12 modèle effectivement vus dans la session — exactement la définition utilisée pour `positives_display` et `positives_relevance`, ce qui rend les trois colonnes directement comparables dans `model_stat_prototype`.

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, DoubleType

# Lignes plates loties, plutot qu'un ArrayType(StructType) construit cote Python. Trois raisons:
#   - un tuple de 4 primitifs pese ~3x moins que les 13 dicts imbriques par trigger de l'ancienne
#     version, qui constituait en plus une seconde copie complete pendant que triggers vivait encore
#   - verifySchema=False supprime la validation ligne par ligne d'un schema imbrique
#   - le lotissement rend la memoire de pointe independante du nombre de triggers, ce qui compte sur
#     un cluster single node ou les 32 Go sont partages entre REPL Python, JVM driver et executeur
# C'est Spark qui reconstruit ensuite le tableau imbrique, via collect_list + sort_array.
MODEL_ROWS_PATH = f"s3://{BUCKET}/{PREFIX}/_staging_model_rows_val_prototype.parquet"
BATCH_TRIGGERS = 20_000

model_rows_schema = StructType([
    StructField("exec_code", LongType(), True),
    StructField("internal_id", LongType(), True),
    StructField("score", DoubleType(), True),
    StructField("rank", IntegerType(), True),
])

keys_schema = StructType([
    StructField("exec_code", LongType(), True),
    StructField("trigger_internal_id", LongType(), True),
])

EMPTY_PRODUCTS = F.array().cast("array<struct<internal_id:bigint,score:double,rank:int>>")


def stage_model_rows(triggers: list[dict], path: str, batch_triggers: int = BATCH_TRIGGERS) -> None:
    """Ecrit les lignes plates par lots. Seul le premier lot ecrase, les suivants ajoutent."""
    mode = "overwrite"

    for start in range(0, len(triggers), batch_triggers):
        rows = [
            (t["exec_code"], v["internalId"], v["score"], v["rank"])
            for t in triggers[start:start + batch_triggers]
            for v in t.get("top12_model", [])
        ]
        if not rows:
            continue

        (
            spark.createDataFrame(rows, schema=model_rows_schema, verifySchema=False)
            .write.mode(mode).parquet(path)
        )
        print(f"  lot {start}-{min(start + batch_triggers, len(triggers))}: {len(rows)} lignes")
        mode = "append"

    if mode == "overwrite":  # aucun trigger scoré: on ecrit quand meme le schema
        spark.createDataFrame([], schema=model_rows_schema).write.mode("overwrite").parquet(path)
        print("  aucun trigger score, parquet vide ecrit")


def build_model_results(triggers: list[dict], sessions_raw: DataFrame) -> DataFrame:

    stage_model_rows(triggers, MODEL_ROWS_PATH)

    # Les cles portent TOUS les triggers, y compris ceux sans categorie ou a pool vide: ils gardent
    # ainsi une ligne avec products_returned vide, comme dans la version precedente.
    df_keys = spark.createDataFrame(
        [(t["exec_code"], t["trigger_internal_id"]) for t in triggers],
        schema=keys_schema,
        verifySchema=False,
    )

    # sort_array trie sur le premier champ du struct, donc rank. Le transform remet ensuite les
    # champs dans l'ordre internal_id, score, rank -- ordre auquel model_stat_prototype se fie via
    # son cast array<struct<internal_id:bigint,score:double,rank:int>>. Ne pas l'inverser.
    df_products = (
        spark.read.parquet(MODEL_ROWS_PATH)
        .groupBy("exec_code")
        .agg(F.sort_array(F.collect_list(F.struct("rank", "internal_id", "score"))).alias("ranked"))
        .withColumn(
            "products_returned",
            F.transform(
                "ranked",
                lambda r: F.struct(
                    r["internal_id"].alias("internal_id"),
                    r["score"].alias("score"),
                    r["rank"].alias("rank"),
                ),
            ),
        )
        .select("exec_code", "products_returned")
    )

    session_ids_by_exec_code = (
        sessions_raw
        .select("exec_code", F.col("session_products.internal_id").alias("session_ids"))
        .dropDuplicates(["exec_code"])
    )

    return (
        df_keys
        .join(df_products, on="exec_code", how="left")
        .join(session_ids_by_exec_code, on="exec_code", how="left")
        .withColumn("products_returned", F.coalesce(F.col("products_returned"), EMPTY_PRODUCTS))
        .withColumn("session_ids", F.coalesce(F.col("session_ids"), F.array().cast("array<bigint>")))
        .withColumn(
            "positives",
            F.filter("products_returned", lambda p: F.array_contains(F.col("session_ids"), p["internal_id"])),
        )
        .withColumn(
            "negatives",
            F.filter("products_returned", lambda p: ~F.array_contains(F.col("session_ids"), p["internal_id"])),
        )
        .select("exec_code", "trigger_internal_id", "positives", "negatives", "products_returned")
    )


model_results_val_prototype = build_model_results(triggers, sessions_raw_val_prototype).cache()

print(f"model_results_val_prototype: {model_results_val_prototype.count()} triggers")
model_results_val_prototype.printSchema()

display(model_results_val_prototype.limit(1))

In [ ]:
model_results_val_prototype.write.mode("overwrite").parquet(
    f"s3://{BUCKET}/{PREFIX}/model_results_val_prototype.parquet"
)
print("model_results_val_prototype.parquet uploaded")

In [ ]:
import gc

# La fusion des deux passes de model_test a fait disparaitre le del/gc.collect() qui separait les
# deux populations, donc la pointe memoire est passee du max des deux a leur somme. On la restaure.
# Un trigger est conserve pour le controle visuel ci-dessous, avant de liberer la liste complete.
sample_trigger = next(t for t in triggers if t.get("top12_model"))

del triggers, candidates_by_category
gc.collect()

print("triggers et candidates_by_category liberes")
print(f"trigger conserve pour le controle visuel: exec_code={sample_trigger['exec_code']}")

## Contrôle visuel sur un trigger

Les trois retours côte à côte pour une même exécution : le top-12 prod dans l'ordre d'affichage, le top-12 prod par `relevanceScore`, et le top-12 du modèle. Les deux variantes prod sont lues depuis le parquet, donc elles reflètent exactement ce qui a servi aux statistiques — plus de recalcul divergent depuis `ads_adlog_fct`.

In [ ]:
def get_customer_db_name(customer_shortname: str) -> str:
    df_customer = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_gold_customer")
        .where(F.col("shortName") == customer_shortname)
        .select(F.col("databaseName").alias("db_name"))
    )
    return df_customer.collect()[0]["db_name"]


def display_product_model(internal_ids: list[int], db_name: str, image_width: int = 150) -> None:
    if not internal_ids:
        print("(aucun produit)")
        return

    ranks_df = spark.createDataFrame(
        [Row(internalId=i, topk_rank=rank) for rank, i in enumerate(internal_ids, start=1)]
    )

    df_products = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_mongo_product_0_current")
        .filter(F.col("db_name") == db_name)
        .select(F.col("internalId").cast("bigint").alias("internalId"), "name", "imageUrl")
        .dropDuplicates(["internalId"])
    )

    df_products_info = (
        ranks_df
        .join(df_products, on="internalId", how="left")
        .select("topk_rank", "internalId", "name", "imageUrl")
        .orderBy("topk_rank")
    )

    for row in df_products_info.collect():
        if row["imageUrl"] is None:
            print(f"{row['topk_rank']}. internalId={row['internalId']} — pas d'image trouvee")
            continue
        display(HTML(f"<p><b>{row['topk_rank']}.</b> {row['name']} (internalId={row['internalId']})</p>"))
        display(HTML(f'<img src="{row["imageUrl"]}" width="{image_width}">'))


db_name = get_customer_db_name("maisons-du-monde")

In [ ]:
# sample_trigger vient de la cellule de liberation memoire ci-dessus
sample_exec_code = sample_trigger["exec_code"]

sample_prod = (
    prod_results_val_prototype
    .filter(F.col("exec_code") == sample_exec_code)
    .collect()[0]
)


def ids_in_rank_order(products) -> list[int]:
    return [int(p["internal_id"]) for p in sorted(products, key=lambda p: p["rank"])]


prod_display_ids = ids_in_rank_order(sample_prod["products_returned_display"])
prod_relevance_ids = ids_in_rank_order(sample_prod["products_returned_relevance"])
model_ids = [int(p["internalId"]) for p in sample_trigger["top12_model"]]

print(f"exec_code {sample_exec_code} -> executionId prod: {sample_prod['execution_id']}")
print(f"produit declencheur: internalId={sample_trigger['trigger_internal_id']}")
print(f"categorie: {sample_trigger['category']} | pool de candidats: {sample_trigger['candidate_pool_size']}")
print()
print(f"prod display  : {prod_display_ids}")
print(f"prod relevance: {prod_relevance_ids}")
print(f"modele        : {model_ids}")
print()
print(f"overlap display/relevance : {len(set(prod_display_ids) & set(prod_relevance_ids))}")
print(f"overlap modele/display    : {len(set(model_ids) & set(prod_display_ids))}")
print(f"overlap modele/relevance  : {len(set(model_ids) & set(prod_relevance_ids))}")

In [ ]:
for title, ids in [
    ("Prod — 12 premiers dans l'ordre d'affichage", prod_display_ids),
    ("Prod — 12 premiers par relevanceScore", prod_relevance_ids),
    ("Modele — top 12", model_ids),
]:
    display(HTML(f"<h3>{title}</h3>"))
    display_product_model(ids, db_name=db_name)